In [ ]:
import os
import gc
import psutil

import logging
import json
import numpy as np
import inspect
import numpy as np
import pandas as pd

import random
from typing import Dict, List, Optional, Tuple, Any

from pathlib import Path
from typing import Optional, Dict, Any




from utils.paths import get_paths
from utils.file_io import save_data


from utils.logging_setup import (
    configure_logging, 
    log_layer_paths,
)

from utils.pipeline_config_loader import (
    load_pipeline_config,
    set_wandb_dir_from_config,
    export_config_snapshot,
    build_truth_config_block,
)

from utils.truths import (
    make_process_run_id,
    initialize_layer_truth,
    update_truth_section,
    build_truth_record,
    save_truth_record,
    append_truth_index,
    stamp_truth_columns,
    load_truth_record_by_hash,
    get_truth_hash,
    get_parent_truth_hash,
    get_pipeline_mode_from_truth,
    get_artifact_path_from_truth,
)


from utils.postgres_util import (
    get_engine_from_env, 
    read_sql_dataframe,
    build_postgres_url, 
    execute_sql,
)

from utils.layer_postgres_writer import (
    write_layer_dataframe, 
    prepare_layer_dataframe,
)






from utils.synthetic.generator.synthetic_profiles import (
    load_rich_profile_csv,
    load_and_merge_rich_profiles,
    load_correlation_pairs_csv,
    load_group_map_csv,
    load_fault_pairings_csv,
)

from utils.synthetic.generator.synthetic_missingness import build_missingness_spec_from_truth_payload
from utils.synthetic.generator.synthetic_export import export_synthetic_batch_to_parquet



from utils.synthetic.generator.synthetic_generator import (
    SyntheticGenerator, 
    EpisodeSpec,
)

from utils.synthetic.generator.synthetic_postgres_writer import (
    ensure_sequence,
    reserve_next_batch_id,
    reserve_cycle_range,
    reset_sequence,
    reset_synthetic_sequences,
    write_stream_batch,
)



from utils.synthetic.pipeline.premelt_stage_writer import (
    build_observations_premelt_stage,
    validate_observations_premelt_stage,
)


from utils.synthetic.pipeline.melt_stage_writer import (
    build_sensor_messages_stage,
    validate_sensor_messages_stage,
)



from utils.synthetic.pipeline.timestamp_stage_writer import (
    ensure_simulation_timing_config_table,
    insert_simulation_timing_config,
    build_sensor_messages_timestamped_stage,
    validate_sensor_messages_timestamped_stage,
)

from utils.synthetic.pipeline.send_queue_stage_writer import (
    build_sensor_messages_send_queue,
    validate_sensor_messages_send_queue,
)


from utils.synthetic.pipeline.producer_queue_manager import (
    ensure_send_queue_runtime_columns,
    ensure_simulation_state_control_table,
    upsert_simulation_state_control,
    read_simulation_state_control,
    claim_pending_send_queue_batch,
    mark_claimed_batch_sent, 
    mark_claimed_batch_failed,
    requeue_failed_messages,
    release_stale_claims,
    get_send_queue_status_counts
)


from utils.synthetic.pipeline.kafka_producer_adapter import (
    run_send_queue_producer_once, 
    run_send_queue_producer_loop, 
    build_sensor_message_payload, 
    json_dumps_safe
)


from utils.synthetic.pipeline.kafka_consumer_adapter import (
    run_kafka_consumer_to_postgres_once, 
    run_kafka_consumer_to_postgres_loop,
)


from utils.synthetic.pipeline.row_rebuilder import (
    rebuild_consumed_messages_to_observations
)

from utils.synthetic.pipeline.rebuild_comparison import (
    build_rebuild_comparison_stage
)


from utils.synthetic.pipeline.synthetic_final_aligned_output_writer import (
    build_synthetic_final_aligned_output_stage
)

from utils.bronze_handoff import (
    handoff_final_aligned_observations_to_bronze, 
    run_bronze_handoff_loop,
)

---

In [ ]:


def memory_gb() -> float:
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / (1024 ** 3)

def log_memory(label: str) -> None:
    print(f"[memory] {label}: {memory_gb():.2f} GB")

---

In [ ]:




SCHEMA = str("capstone")

DATASET_ID = str("pump_synthetic_v1")
RUN_ID = str("premelt_run_001")
ASSET_ID = str("pump_asset_001")

IF_EXISTS_FLAG = str("replace")
RANDOM_SEED = int(42)
NUMBER_OF_SENSORS = int(52)
CHUNK_SIZE = int(10000)

SEND_QUEUE_MODE = "LOOP"
CONSUMER_BATCH_MODE = "LOOP"



'''
# Generator
# --- Notebook params ---
STAGE = "synthetic"
DATASET = "pump"
MODE = "train"
PROFILE = "default"

# ---- Mode switch ----
MODE = "batch"         # "single" | "batch"
TARGET_ROWS = 72_000
MAX_EPISODES = 1_000_000  # safety cap

# ---- policy knobs ----
EPISODE_MAX_ROWS = 3_000   # prevents monster episodes; forces multiple episodes in a 10k batch
ALLOW_OVERSHOOT = False    # if True, can overshoot when remaining can't fit minimum core

# ---- failure rarity control (match real dataset frequency) ----
# Real dataset: ~7 failures per 250,000 rows  => ~1 failure per 35,714 rows
ROWS_PER_FAILURE = 250_000 / 7  # ~35714.2857

'''


# Premelt - Stamping
PREMELT_SOURCE_TABLE_NAME = str("synthetic_pump_stream")
PREMELT_DESTINATION_TABLE_NAME = str("synthetic_observations_premelt_stage")

# Melting
MELT_SOURCE_TABLE_NAME = str("synthetic_observations_premelt_stage")
MELT_DESTINATION_TABLE_NAME = str("synthetic_sensor_messages_stage")

# Timestamping
TIMESTAMP_CHUNK_SIZE = int(250000)

SIMULATION_TIME_CONFIG_TABLE_NAME = str("simulation_timing_config")
SIMULATION_START_DATETIME = str("2026-03-19 08:00:00+00:00")
SIMULATION_SAMPLING_INTERVAL_SECONDS = float(60.0)

TIMESTAMPED_SOURCE_TABLE_NAME = str("synthetic_sensor_messages_stage")
TIMESTAMPED_DESTINATION_TABLE_NAME = str("synthetic_sensor_messages_timestamped_stage")

# Build Send Queue
QUEUE_STATUS_DEFAULT = str("pending")
SEND_QUEUE_CHUNK_SIZE = int(250000)

SEND_QUEUE_SOURCE_TABLE_NAME = str("synthetic_sensor_messages_timestamped_stage")
SEND_QUEUE_DESTINATION_TABLE_NAME = str("synthetic_sensor_messages_send_queue")

# Producer Queue Manager
IS_ENABLED_FLAG = True

PRODUCER_TOPIC = str("pump.telemetry.synthetic")
PRODUCER_BATCH_SIZE = int(5200)
PRODUCER_POLL_SECONDS = float(0.0)
PRODUCER_MAX_SEND_ATTEMPTS = int(3)

PRODUCER_WORKER_ID = str("producer_worker_001")

SIMULATION_TABLE_NAME = str("simulation_state_control")
SEND_QUEUE_TABLE_NAME = str("synthetic_sensor_messages_send_queue")

CLAIM_BATCH_SIZE = int(500)
STALE_CLAIM_RELEASE_MINUTES = int(15)

# Producer Adapter
CLIENT_ID = str("pump-telemetry-producer")
FLUSH_TIMEOUT_SECONDS = float(30.0)

MAX_BATCHES = int(10)
STOP_ON_FAILURE_FLAG = True


# Consumer Adapter
KAFKA_TOPIC = str("pump.telemetry.synthetic")
CONSUMER_DESTINATION_TABLE_NAME = str("synethic_sensor_messages_consumed_stage")
CONSUMER_GROUP_ID = str("synthetic-telemetry-consumer-group")
CONSUMER_WORKER_ID = str("consumer_worker_001")

CONSUMER_MAX_MESSAGES = int(520)
CONSUMER_POLL_TIMEOUT_SECONDS = float(1.0)

AUTO_OFFSET_RESET = str("earliest")

IDLE_SLEEP_SECONDS = float(0.0)
STOP_ON_EMPTY_FLAG = True


# Rebuild Row
REBUILD_CONSUMED_MESSAGES_SOURCE_TABLE_NAME = str("synthetic_sensor_messages_consumed_stage")
REBUILT_CONSUMED_MESSAGES_DESTINATION_TABLE_NAME = str("synthetic_sensor_observations_rebuilt_stage")

REBUILD_STATUS = str("pending")

COMPLETE_ONLY_FLAG = True
MARK_SOURCE_REBUILT_FLAG = True

OBSERVATION_WINDOW_SIZE = int(2500)


# Rebuild Comparison

FLOAT_TOLERANCE = 1e-9

PREMELT_SOURCE_TABLE_NAME = str("synthetic_observations_premelt_stage")
REBUILT_DESTINATION_TABLE_NAME = str("synthetic_sensor_observations_rebuilt_stage")
TARGET_TABLE_NAME = str("synthetic_sensor_rebuild_comparison_stage")

rebuild_observation_to_check = 1

#Build Final Aligned Observation
REBUILT_SOURCE_TABLE_NAME = str("synthetic_sensor_observations_rebuilt_stage")
TARGET_TABLE_NAME = str("synthetic_sensor_observations_final_output")

TIMESTAMP_SOURCE_PRIORITY = (
    "observation_timestamp",
    "timestamp",
    "created_at",
)

STATUS_SOURCE_PRIORITY = (
    "machine_status",
    "stream_state",
    "phase",
)

STATUS_MAPPING = {
    "normal": "NORMAL",
    "broken": "BROKEN",
    "abnormal": "BROKEN",
    "failure": "BROKEN",
    "failed": "BROKEN",
    "fault": "BROKEN",
    "recovering": "RECOVERING",
    "recovery": "RECOVERING",
}

# Bronze Handoff
# Script style

# Mode
# "row" | "row_batch" | "full_batch"

run_mode = "row_batch"

---

In [ ]:
# Create Engine
engine = get_engine_from_env()


---

In [ ]:

premelt_table_name = build_observations_premelt_stage(
    engine=engine,
    schema=SCHEMA,
    source_table=PREMELT_SOURCE_TABLE_NAME,
    target_table=PREMELT_DESTINATION_TABLE_NAME,
    dataset_id=DATASET_ID,
    run_id=RUN_ID,
    asset_id=ASSET_ID,
    if_exists=IF_EXISTS_FLAG,
)

print("Built table:", premelt_table_name)

In [ ]:

observation_validation_dataframe = validate_observations_premelt_stage(
    engine=engine,
    schema=SCHEMA,
    table_name=PREMELT_DESTINATION_TABLE_NAME,
)

display(observation_validation_dataframe)

In [ ]:
# Inspection

inspection_dataframe = read_sql_dataframe(
    engine,
    """
    SELECT
        dataset_id,
        run_id,
        asset_id,
        generated_row_id,
        observation_index,
        batch_id,
        row_in_batch,
        global_cycle_id,
        stream_state,
        phase,
        meta_episode_id,
        is_telemetry_event,
        telemetry_event_type,
        producer_send_attempt
    FROM capstone.synthetic_observations_premelt_stage
    ORDER BY observation_index
    LIMIT 25
    """
)

display(inspection_dataframe)

In [ ]:
# Dispose Engine
engine.dispose()

---

In [ ]:
# Create Engine
engine = get_engine_from_env()

In [ ]:

ensure_simulation_timing_config_table(
    engine=engine,
    schema=SCHEMA,
    table_name=SIMULATION_TIME_CONFIG_TABLE_NAME,
)


In [ ]:

insert_simulation_timing_config(
    engine=engine,
    dataset_id=DATASET_ID,
    run_id=RUN_ID,
    simulation_start_datetime=SIMULATION_START_DATETIME,
    sampling_interval_seconds=SIMULATION_SAMPLING_INTERVAL_SECONDS,
    schema=SCHEMA,
    table_name=SIMULATION_TIME_CONFIG_TABLE_NAME,
    set_active=True,
    deactivate_existing_for_run=True,
)

print("Timing config ready.")

In [ ]:
timestamped_table_name = build_sensor_messages_timestamped_stage(
    engine=engine,
    schema=SCHEMA,
    source_table=TIMESTAMPED_SOURCE_TABLE_NAME,
    target_table=TIMESTAMPED_DESTINATION_TABLE_NAME,
    timing_config_table=SIMULATION_TIME_CONFIG_TABLE_NAME,
    dataset_id=DATASET_ID,
    run_id=RUN_ID,
    if_exists=IF_EXISTS_FLAG,
    chunk_size=TIMESTAMP_CHUNK_SIZE,
)

print("Built table:", timestamped_table_name)


In [ ]:
timestamp_validation_dataframe = validate_sensor_messages_timestamped_stage(
    engine=engine,
    schema=SCHEMA,
    table_name=TIMESTAMPED_DESTINATION_TABLE_NAME,
)

display(timestamp_validation_dataframe)

In [ ]:

timetstamp_sample_dataframe = read_sql_dataframe(
    engine,
    """
    SELECT
        observation_index,
        observation_timestamp,
        message_sequence_index,
        sensor_name,
        sensor_index,
        sensor_value,
        stream_state,
        phase,
        meta_episode_id
    FROM capstone.synthetic_sensor_messages_timestamped_stage
    ORDER BY observation_index, message_sequence_index, sensor_index
    LIMIT 104
    """
)

display(timetstamp_sample_dataframe)

In [ ]:
timestamp_check_dataframe = read_sql_dataframe(
    engine,
    """
    SELECT
        observation_index,
        COUNT(*) AS message_count,
        COUNT(DISTINCT observation_timestamp) AS distinct_timestamps_in_observation,
        MIN(observation_timestamp) AS observation_timestamp_min,
        MAX(observation_timestamp) AS observation_timestamp_max
    FROM capstone.synthetic_sensor_messages_timestamped_stage
    GROUP BY observation_index
    ORDER BY observation_index
    LIMIT 10
    """
)

display(timestamp_check_dataframe)

In [ ]:
# Dispose Engine
engine.dispose()

----

In [ ]:
# Create Engine
engine = get_engine_from_env()

In [ ]:
log_memory("before read")
# read chunk
log_memory("after read")
# melt chunk
log_memory("after melt")
# write chunk
log_memory("after write")
gc.collect()
log_memory("after gc")

In [ ]:

melt_table_name = build_sensor_messages_stage(
    engine=engine,
    schema=SCHEMA,
    source_table=MELT_SOURCE_TABLE_NAME,
    target_table=MELT_DESTINATION_TABLE_NAME,
    if_exists=IF_EXISTS_FLAG,
    random_seed=RANDOM_SEED,
    n_sensors=NUMBER_OF_SENSORS,
    chunk_size=CHUNK_SIZE,
)

In [ ]:
log_memory("before read")
# read chunk
log_memory("after read")
# melt chunk
log_memory("after melt")
# write chunk
log_memory("after write")
gc.collect()
log_memory("after gc")

In [ ]:
print("Built table:", melt_table_name)


In [ ]:

melt_validation_dataframe = validate_sensor_messages_stage(
    engine=engine,
    schema=SCHEMA,
    table_name=MELT_DESTINATION_TABLE_NAME,
)

display(melt_validation_dataframe)

In [ ]:
melt_sample_dataframe = read_sql_dataframe(
    engine,
    """
    SELECT
        observation_index,
        generated_row_id,
        sensor_name,
        sensor_index,
        sensor_value,
        message_sequence_index,
        stream_state,
        phase,
        meta_episode_id
    FROM capstone.synthetic_sensor_messages_stage
    ORDER BY observation_index, message_sequence_index
    LIMIT 104
    """
)

display(melt_sample_dataframe)

In [ ]:
sequence_check_dataframe = read_sql_dataframe(
    engine,
    """
    SELECT
        observation_index,
        COUNT(*) AS message_count,
        COUNT(DISTINCT sensor_index) AS distinct_sensor_count,
        MIN(message_sequence_index) AS min_msg_seq,
        MAX(message_sequence_index) AS max_msg_seq,
        COUNT(DISTINCT message_sequence_index) AS distinct_msg_seq_count
    FROM capstone.synthetic_sensor_messages_stage
    GROUP BY observation_index
    ORDER BY observation_index
    LIMIT 10
    """
)

display(sequence_check_dataframe)

In [ ]:
# Dispose Engine
engine.dispose()

---

In [ ]:
# Create Engine
engine = get_engine_from_env()

In [ ]:
send_queue_table_name = build_sensor_messages_send_queue(
    engine=engine,
    schema=SCHEMA,
    source_table=SEND_QUEUE_SOURCE_TABLE_NAME,
    target_table=SEND_QUEUE_DESTINATION_TABLE_NAME,
    if_exists=IF_EXISTS_FLAG,
    queue_status_default=QUEUE_STATUS_DEFAULT,
    chunk_size=SEND_QUEUE_CHUNK_SIZE,
)

print("Built table:", send_queue_table_name)

In [ ]:

send_queue_validation_dataframe = validate_sensor_messages_send_queue(
    engine=engine,
    schema=SCHEMA,
    table_name=SEND_QUEUE_DESTINATION_TABLE_NAME,
)

display(send_queue_validation_dataframe)

In [ ]:
send_queue_sample_dataframe = read_sql_dataframe(
    engine,
    """
    SELECT
        message_key,
        observation_index,
        observation_timestamp,
        message_sequence_index,
        sensor_name,
        sensor_index,
        sensor_value,
        queue_status,
        queued_at,
        producer_sent_at
    FROM capstone.synthetic_sensor_messages_send_queue
    ORDER BY observation_index, message_sequence_index, sensor_index
    LIMIT 104
    """
)

display(send_queue_sample_dataframe)

In [ ]:
pending_dataframe = read_sql_dataframe(
    engine,
    """
    SELECT *
    FROM capstone.synthetic_sensor_messages_send_queue
    WHERE queue_status = 'pending'
      AND producer_sent_at IS NULL
    ORDER BY observation_index, message_sequence_index, sensor_index
    LIMIT 500
    """
)

display(pending_dataframe.head(100))

In [ ]:
# Dispose Engine
engine.dispose()

---

In [ ]:
# Create Engine
engine = get_engine_from_env()

In [ ]:

ensure_send_queue_runtime_columns(
    engine=engine,
    schema = SCHEMA,
    table_name=SEND_QUEUE_TABLE_NAME,
)


In [ ]:

ensure_simulation_state_control_table(
    engine=engine,
    schema=SCHEMA,
    table_name=SIMULATION_TABLE_NAME,
)

In [ ]:

upsert_simulation_state_control(
    engine=engine,
    dataset_id = DATASET_ID,
    run_id = RUN_ID,
    is_enabled = IS_ENABLED_FLAG,
    producer_topic = PRODUCER_TOPIC,
    producer_batch_size = PRODUCER_BATCH_SIZE,
    producer_poll_seconds = PRODUCER_POLL_SECONDS,
    max_send_attempts = PRODUCER_MAX_SEND_ATTEMPTS,
    schema = SCHEMA,
    table_name=SIMULATION_TABLE_NAME,
)


In [ ]:

control_row = read_simulation_state_control(
    engine=engine,
    dataset_id=DATASET_ID,
    run_id=RUN_ID,
    schema=SCHEMA,
    table_name=SIMULATION_TABLE_NAME,
)

display(control_row)

In [ ]:
# Dispose Engine
engine.dispose()

----

In [ ]:
# Create Engine
engine = get_engine_from_env()

In [ ]:

if SEND_QUEUE_MODE == "SINGLE":

    result = run_send_queue_producer_once(
        engine=engine,
        dataset_id=DATASET_ID,
        run_id=RUN_ID,
        schema=SCHEMA,
        queue_table=SEND_QUEUE_TABLE_NAME,
        control_table=SIMULATION_TABLE_NAME,
        producer_worker_id=PRODUCER_WORKER_ID,
        client_id=CLIENT_ID,
        flush_timeout_seconds=FLUSH_TIMEOUT_SECONDS,
    )

    display(result)

elif SEND_QUEUE_MODE == "LOOP":

    results = run_send_queue_producer_loop(
        engine=engine,
        dataset_id=DATASET_ID,
        run_id=RUN_ID,
        schema=SCHEMA,
        queue_table=SEND_QUEUE_TABLE_NAME,
        control_table=SIMULATION_TABLE_NAME,
        producer_worker_id=PRODUCER_WORKER_ID,
        client_id=CLIENT_ID,
        max_batches=MAX_BATCHES,
        stop_on_failure=STOP_ON_FAILURE_FLAG,
        flush_timeout_seconds=FLUSH_TIMEOUT_SECONDS,
    )

    display(results)

else:
    raise ValueError(f"SEND_QUEUE_MODE: {SEND_QUEUE_MODE} selected is not an available option, please select SINGLE or LOOP")



In [ ]:

preview_dataframe = read_sql_dataframe(
    engine,
    """
    SELECT *
    FROM capstone.synthetic_sensor_messages_send_queue
    ORDER BY observation_index, message_sequence_index, sensor_index
    LIMIT 1
    """
)

payload = build_sensor_message_payload(preview_dataframe.iloc[0].to_dict())
print(json_dumps_safe(payload))

In [ ]:
# Dispose Engine
engine.dispose()

----

In [ ]:
# Create Engine
engine = get_engine_from_env()

In [ ]:


rebuild_result = rebuild_consumed_messages_to_observations(
    engine=engine,
    schema=SCHEMA,
    source_table=REBUILD_CONSUMED_MESSAGES_SOURCE_TABLE_NAME,
    target_table=REBUILT_CONSUMED_MESSAGES_DESTINATION_TABLE_NAME,
    dataset_id=DATASET_ID,
    run_id=RUN_ID,
    rebuild_status=REBUILD_STATUS,
    n_sensors=NUMBER_OF_SENSORS,
    complete_only=COMPLETE_ONLY_FLAG,
    mark_source_rebuilt=MARK_SOURCE_REBUILT_FLAG,
    observation_window_size=OBSERVATION_WINDOW_SIZE,
)

display(rebuild_result)



In [ ]:
rebuild_validation_dataframe = read_sql_dataframe(
    engine,
    """
    SELECT
        COUNT(*) AS rebuilt_row_count,
        COUNT(*) FILTER (WHERE rebuild_is_complete = TRUE) AS complete_row_count,
        MIN(observation_index) AS min_observation_index,
        MAX(observation_index) AS max_observation_index,
        MIN(observation_timestamp) AS min_observation_timestamp,
        MAX(observation_timestamp) AS max_observation_timestamp
    FROM capstone.synthetic_sensor_observations_rebuilt_stage
    """
)

display(rebuild_validation_dataframe)

In [ ]:
rebuild_sample_dataframe = read_sql_dataframe(
    engine,
    """
    SELECT
        dataset_id,
        run_id,
        asset_id,
        observation_index,
        observation_timestamp,
        stream_state,
        phase,
        meta_episode_id,
        meta_primary_fault_type,
        sensor_00,
        sensor_01,
        sensor_02,
        sensor_03,
        sensor_04,
        rebuild_sensor_count,
        rebuild_is_complete
    FROM capstone.synthetic_sensor_observations_rebuilt_stage
    ORDER BY observation_index
    LIMIT 10
    """
)

display(rebuild_sample_dataframe)


In [ ]:
rebuild_incomplete_dataframe = read_sql_dataframe(
    engine,
    """
    SELECT
        dataset_id,
        run_id,
        asset_id,
        observation_index,
        rebuild_sensor_count,
        rebuild_is_complete,
        rebuild_notes
    FROM capstone.synthetic_sensor_observations_rebuilt_stage
    WHERE rebuild_is_complete = FALSE
    ORDER BY observation_index
    LIMIT 25
    """
)

display(rebuild_incomplete_dataframe)

In [ ]:
# Dispose Engine
engine.dispose()

----

In [ ]:
# Create Engine
engine = get_engine_from_env()

In [ ]:
comparison_result = build_rebuild_comparison_stage(
    engine=engine,
    schema=SCHEMA,
    premelt_table=PREMELT_SOURCE_TABLE_NAME,
    rebuilt_table=REBUILT_DESTINATION_TABLE_NAME,
    target_table=TARGET_TABLE_NAME,
    dataset_id=DATASET_ID,
    run_id=RUN_ID,
    n_sensors=NUMBER_OF_SENSORS,
    float_tolerance=FLOAT_TOLERANCE,
    observation_window_size=OBSERVATION_WINDOW_SIZE,
)

display(comparison_result)

In [ ]:
comparison_summary_dataframe = read_sql_dataframe(
    engine,
    """
    SELECT
        COUNT(*) AS comparison_row_count,
        COUNT(*) FILTER (WHERE comparison_all_fields_match = TRUE) AS all_match_count,
        COUNT(*) FILTER (WHERE comparison_all_fields_match = FALSE) AS mismatch_count,
        MAX(comparison_mismatch_count) AS max_mismatch_count
    FROM capsone.synthetic_sensor_rebuild_comparison_stage
    """
)

display(comparison_summary_dataframe)


In [ ]:
comparison_mismatch_dataframe = read_sql_dataframe(
    engine,
    """
    SELECT
        dataset_id,
        run_id,
        asset_id,
        observation_index,
        comparison_mismatch_count,
        comparison_notes,
        exists_in_original,
        exists_in_rebuilt,
        rebuilt__rebuild_sensor_count,
        rebuilt__rebuild_is_complete
    FROM capstone.synthetic_sensor_rebuild_comparison_stage
    WHERE comparison_all_fields_match = FALSE
    ORDER BY observation_index
    LIMIT 50
    """
)

display(comparison_mismatch_dataframe)

In [ ]:


rebuild_detail_dataframe = read_sql_dataframe(
    engine,
    f"""
    SELECT *
    FROM capstone.synthetic_sensor_rebuild_comparison_stage
    WHERE observation_index = {int(rebuild_observation_to_check)}
    """
)


display(rebuild_detail_dataframe)

In [ ]:
# Dispose Engine
engine.dispose()

---

In [ ]:
# Create Engine
engine = get_engine_from_env()

In [ ]:
final_output_result = build_synthetic_final_aligned_output_stage(
    engine=engine,
    schema=SCHEMA,
    rebuilt_table=REBUILT_SOURCE_TABLE_NAME,
    target_table=TARGET_TABLE_NAME,
    dataset_id=DATASET_ID,
    run_id=RUN_ID,
    n_sensors=NUMBER_OF_SENSORS,
    complete_only=COMPLETE_ONLY_FLAG,
    if_exists=IF_EXISTS_FLAG,
    observation_window_size=OBSERVATION_WINDOW_SIZE,
    timestamp_source_priority=TIMESTAMP_SOURCE_PRIORITY,
    status_source_priority=STATUS_SOURCE_PRIORITY,
    status_mapping=STATUS_MAPPING,
    timestamp_output_column="timestamp",
    machine_status_output_column="machine_status",
)

display(final_output_result)

In [ ]:
final_output_validation_dataframe = read_sql_dataframe(
    engine,
    f"""
    SELECT
        COUNT(*) AS row_count,
        COUNT(DISTINCT dataset_id) AS dataset_id_count,
        COUNT(DISTINCT run_id) AS run_id_count,
        COUNT(DISTINCT asset_id) AS asset_id_count,
        MIN(timestamp) AS min_timestamp,
        MAX(timestamp) AS max_timestamp,
        COUNT(*) FILTER (WHERE machine_status = 'NORMAL') AS normal_rows,
        COUNT(*) FILTER (WHERE machine_status = 'BROKEN') AS broken_rows,
        COUNT(*) FILTER (WHERE machine_status = 'RECOVERING') AS recovering_rows
    FROM {SCHEMA}.{TARGET_TABLE_NAME}
    """
)

display(final_output_validation_dataframe)


In [ ]:
final_output_sample_dataframe = read_sql_dataframe(
    engine,
    f"""
    SELECT
        dataset_id,
        run_id,
        asset_id,
        timestamp,
        sensor_00,
        sensor_01,
        sensor_02,
        sensor_03,
        sensor_04,
        machine_status
    FROM {SCHEMA}.{TARGET_TABLE_NAME}
    ORDER BY timestamp, dataset_id, run_id, asset_id
    LIMIT 10
    """
)

display(final_output_sample_dataframe)


In [ ]:
final_output_status_distribution_dataframe = read_sql_dataframe(
    engine,
    f"""
    SELECT
        machine_status,
        COUNT(*) AS row_count
    FROM {SCHEMA}.{TARGET_TABLE_NAME}
    GROUP BY machine_status
    ORDER BY machine_status
    """
)

display(final_output_status_distribution_dataframe)


In [ ]:
# Dispose Engine
engine.dispose()

---

In [ ]:
# Create Engine
engine = get_engine_from_env()

In [ ]:

bronze_handoff_results = run_bronze_handoff_loop(
    engine=engine,
    mode=run_mode,          # "row" | "row_batch" | "full_batch"
    batch_size=500,
    schema="capstone",
    source_table="synthetic_sensor_observations_final_aligned_stage",
    target_table="bronze_observations_input_stage",
    dataset_id="pump_synthetic_v1",
    run_id="premelt_run_001",
    complete_only=True,
    max_iterations=None,
    stop_on_failure=True,
)

print(bronze_handoff_results[-1] if bronze_handoff_results else {"status": "no_run"})

In [ ]:
handoff_final_aligned_observation_result = handoff_final_aligned_observations_to_bronze(
    engine=engine,
    mode="full_batch",
    batch_size=500,  # ignored in full_batch
    schema="capstone",
    source_table="synthetic_sensor_observations_final_aligned_stage",
    target_table="bronze_observations_input_stage",
    dataset_id="pump_synthetic_v1",
    run_id="premelt_run_001",
    complete_only=True,
)

display(handoff_final_aligned_observation_result)


In [ ]:
# Dispose Engine
engine.dispose()